In [ ]:
import sys
import os
import json

sys.path.append('/kaggle/input/datasets/nellytsiblaki/slplab3/slplab3')



In [ ]:
!pip install evaluate transformers datasets scikit-learn

In [ ]:
import numpy as np
import evaluate
from datasets import Dataset
from transformers import TrainingArguments, Trainer, AutoTokenizer, AutoModelForSequenceClassification
from sklearn.preprocessing import LabelEncoder
from utils.load_datasets import load_MR, load_Semeval2017A
import torch
import gc
from transformers import EarlyStoppingCallback, AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

In [ ]:
#define datasets, models and metrics
DATASET = 'Semeval2017A' # 'MR' or 'Semeval2017A'
PRETRAINED_MODEL_MR = ['distilbert-base-uncased', 'albert-base-v2', 'google/electra-small-discriminator']
PRETRAINED_MODEL_SEM= ['distilroberta-base', 'distilbert-base-uncased']

metrics = evaluate.combine(["accuracy", "recall", "f1"])
accuracy_metric = evaluate.load("accuracy")
recall_metric = evaluate.load("recall")
f1_metric = evaluate.load("f1")

In [ ]:


def compute_metrics_mr(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metrics.compute(predictions=predictions, references=labels)


def compute_metrics_sem(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    
    acc = accuracy_metric.compute(predictions=predictions, references=labels)
    
    rec = recall_metric.compute(predictions=predictions, references=labels, average='weighted')
    f1 = f1_metric.compute(predictions=predictions, references=labels, average='weighted')
    
    return {
        "accuracy": acc["accuracy"],
        "recall": rec["recall"],
        "f1": f1["f1"]
    }

In [ ]:
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

In [ ]:
def prepare_dataset(X, y):
    texts, labels = [], []
    for text, label in zip(X, y):
        texts.append(text)
        labels.append(label)

    return Dataset.from_dict({'text': texts, 'label': labels})

In [ ]:
# load the raw data
if DATASET == "Semeval2017A":
    X_train, y_train, X_test, y_test = load_Semeval2017A()
elif DATASET == "MR":
    X_train, y_train, X_test, y_test = load_MR()
else:
    raise ValueError("Invalid dataset")

In [ ]:
# encode labels
le = LabelEncoder()
le.fit(list(set(y_train)))
y_train = le.transform(y_train)
y_test = le.transform(y_test)
n_classes = len(list(le.classes_))

In [ ]:
# prepare datasets
train_set = prepare_dataset(X_train, y_train)
test_set = prepare_dataset(X_test, y_test)

In [ ]:
batch_sizes = [8, 16]
epochs = 5
learning_rates = [2e-5, 5e-5]


In [ ]:

for PRETRAINED_MODEL in PRETRAINED_MODEL_MR:
  #define tokenizer
  tokenizer = AutoTokenizer.from_pretrained(PRETRAINED_MODEL)

  # tokenize datasets
  tokenized_train_set = train_set.map(tokenize_function)
  tokenized_test_set = test_set.map(tokenize_function)

  for batch in batch_sizes:
    for lr in learning_rates:

      # define model
      model = AutoModelForSequenceClassification.from_pretrained(
          PRETRAINED_MODEL, num_labels=n_classes)

      # TODO: Main-lab-Q7 - customize hyperparameters once you are ready to execute on a GPU
      # training setup
        
      args = TrainingArguments(
              output_dir="output",
              eval_strategy="epoch",
              save_strategy="epoch",
              num_train_epochs=epochs,
              per_device_train_batch_size=batch,
              per_device_eval_batch_size=batch,
              learning_rate = lr,
              fp16=True,
              load_best_model_at_end=True, 
              metric_for_best_model="f1",   
              save_total_limit=1,           
              report_to="none"
      )
        
      trainer = Trainer(
          model=model,
          args=args,
          train_dataset=tokenized_train_set,
          eval_dataset=tokenized_test_set,
          compute_metrics=compute_metrics_mr
      )

      print(f"Training {PRETRAINED_MODEL} with batch size = {batch}, learning rate = {lr}")
      # train
      trained_model = trainer.train()

      
      final_metrics = trainer.evaluate()
    
    
      result_entry = {
        "model": PRETRAINED_MODEL,
        "batch_size": batch,
        "learning_rate": lr,
        "best_eval_accuracy": round(final_metrics.get("eval_accuracy", 0), 4),
        "best_eval_f1": round(final_metrics.get("eval_f1", 0), 4),
        "best_eval_recall": round(final_metrics.get("eval_recall", 0), 4)
      }

      with open("best_models.txt", "a") as f:
          f.write(json.dumps(result_entry) + "\n")

      del model
      del trainer
      gc.collect()
      torch.cuda.empty_cache()



In [ ]:
for PRETRAINED_MODEL in PRETRAINED_MODEL_SEM:
  #define tokenizer
  tokenizer = AutoTokenizer.from_pretrained(PRETRAINED_MODEL)

  # tokenize datasets
  tokenized_train_set = train_set.map(tokenize_function)
  tokenized_test_set = test_set.map(tokenize_function)

  for batch in batch_sizes:
    for lr in learning_rates:

      # define model
      model = AutoModelForSequenceClassification.from_pretrained(
          PRETRAINED_MODEL, num_labels=n_classes)

      # TODO: Main-lab-Q7 - customize hyperparameters once you are ready to execute on a GPU
      # training setup
        
      args = TrainingArguments(
              output_dir="output",
              eval_strategy="epoch",
              save_strategy="epoch",
              num_train_epochs=epochs,
              per_device_train_batch_size=batch,
              per_device_eval_batch_size=batch,
              learning_rate = lr,
              fp16=True,
              load_best_model_at_end=True, 
              metric_for_best_model="f1",   
              save_total_limit=1,           
              report_to="none"
      )
        
      trainer = Trainer(
          model=model,
          args=args,
          train_dataset=tokenized_train_set,
          eval_dataset=tokenized_test_set,
          compute_metrics=compute_metrics_sem
      )

      print(f"Training {PRETRAINED_MODEL} with batch size = {batch}, learning rate = {lr}")
      
      # train
      trained_model = trainer.train()

      
      final_metrics = trainer.evaluate()
    
    
      result_entry = {
        "model": PRETRAINED_MODEL,
        "batch_size": batch,
        "learning_rate": lr,
        "best_eval_accuracy": round(final_metrics.get("eval_accuracy", 0), 4),
        "best_eval_f1": round(final_metrics.get("eval_f1", 0), 4),
        "best_eval_recall": round(final_metrics.get("eval_recall", 0), 4)
      }

      with open("semeval_best_models.txt", "a") as f:
          f.write(json.dumps(result_entry) + "\n")

      del model
      del trainer
      gc.collect()
      torch.cuda.empty_cache()


In [ ]:
def tokenize_function_tweet(examples):
    return tokenizer(examples["text"], padding="max_length",
        truncation=True, max_length=512)

In [ ]:
PRETRAINED_MODEL = 'cardiffnlp/twitter-roberta-base-sentiment'

#define tokenizer
tokenizer = AutoTokenizer.from_pretrained(PRETRAINED_MODEL)

# tokenize datasets
tokenized_train_set = train_set.map(tokenize_function_tweet)
tokenized_test_set = test_set.map(tokenize_function_tweet)

for batch in batch_sizes:
        
    for lr in learning_rates:

  # define model
        model = AutoModelForSequenceClassification.from_pretrained(
        PRETRAINED_MODEL, num_labels=n_classes)

  # TODO: Main-lab-Q7 - customize hyperparameters once you are ready to execute on a GPU
  # training setup
    
        args = TrainingArguments(
                  output_dir="output",
                  eval_strategy="epoch",
                  save_strategy="epoch",
                  num_train_epochs=epochs,
                  per_device_train_batch_size=batch,
                  per_device_eval_batch_size=batch,
                  learning_rate = lr,
                  fp16=True,
                  load_best_model_at_end=True, 
                  metric_for_best_model="f1",   
                  save_total_limit=1,           
                  report_to="none"
              )
        
        trainer = Trainer(
              model=model,
              args=args,
              train_dataset=tokenized_train_set,
              eval_dataset=tokenized_test_set,
              compute_metrics=compute_metrics_sem
          )
        
        print(f"Training {PRETRAINED_MODEL} with batch size = {batch}, learning rate = {lr}")
      
      # train
        trained_model = trainer.train()
    
          
        final_metrics = trainer.evaluate()


        result_entry = {
            "model": PRETRAINED_MODEL,
            "batch_size": batch,
            "learning_rate": lr,
            "best_eval_accuracy": round(final_metrics.get("eval_accuracy", 0), 4),
            "best_eval_f1": round(final_metrics.get("eval_f1", 0), 4),
            "best_eval_recall": round(final_metrics.get("eval_recall", 0), 4)
        }
    
        with open("roberta_best_models.txt", "a") as f:
            f.write(json.dumps(result_entry) + "\n")
    
        del model
        del trainer
        gc.collect()
        torch.cuda.empty_cache()